# 02 - Data Cleaning and Feature Preparation

This notebook prepares the collected Old School RuneScape Hiscores dataset for later similarity analysis and clustering.

The main goal is to transform the raw Hiscores output into a clean object-variable data matrix, where:

- each row represents one OSRS player,
- each column represents a numeric attribute,
- the selected attributes describe the player's skill profile.

This step is necessary before applying distance-based and clustering methods.

## 1. Package imports

In this section, we import the Python packages required for data loading, cleaning and basic feature preparation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Project paths

The cleaned dataset will be read from and saved into the `data/processed` folder.

The previous notebook created the first processed CSV file from the collected Hiscores data.

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

input_path = PROCESSED_DATA_DIR / "osrs_hiscores_auto_sample.csv"
output_path = PROCESSED_DATA_DIR / "osrs_hiscores_cleaned.csv"

input_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_auto_sample.csv')

## 3. Load the collected Hiscores dataset

The dataset contains public Hiscores data for selected OSRS players.

At this stage, the dataset contains several types of columns:

- player name,
- skill rank,
- skill level,
- skill experience.

For clustering, we will initially focus on skill level values, because they describe player progression in a simple and interpretable way.

In [3]:
df = pd.read_csv(input_path)

df.head()

,player,overall_rank,overall_level,overall_xp,attack_rank,attack_level,attack_xp,defence_rank,defence_level,defence_xp,...,runecraft_xp,hunter_rank,hunter_level,hunter_xp,construction_rank,construction_level,construction_xp,sailing_rank,sailing_level,sailing_xp
0,Obbyy,3691,2376,1059127090,60272,99,25735539,15816,99,32203282,...,13514378,5521,99,42192236,20495,99,13566811,33649,99,13057617
1,Obby Cape,23851,2376,537134823,133254,99,18904741,36445,99,24819279,...,13041818,87418,99,13245677,27973,99,13449454,11248,99,14406893
2,Obby Apples,104604,2278,950095206,101045,99,21090532,136,99,200000000,...,13297316,114708,99,13089257,93260,99,13142401,-1,1,0
3,Obby Kenobi,1009395,1752,290653472,-1,1,40,-1,1,0,...,752885,350534,85,3517479,852014,75,1229304,1496,99,30442470
4,obbE x,25337,2376,521433553,220336,99,15566135,47533,99,22895521,...,13083862,105571,99,13123773,47912,99,13294394,45572,99,13043708


In [4]:
df.shape

(808, 76)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 808 entries, 0 to 807
Data columns (total 76 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   player              808 non-null    object
 1   overall_rank        808 non-null    int64 
 2   overall_level       808 non-null    int64 
 3   overall_xp          808 non-null    int64 
 4   attack_rank         808 non-null    int64 
 5   attack_level        808 non-null    int64 
 6   attack_xp           808 non-null    int64 
 7   defence_rank        808 non-null    int64 
 8   defence_level       808 non-null    int64 
 9   defence_xp          808 non-null    int64 
 10  strength_rank       808 non-null    int64 
 11  strength_level      808 non-null    int64 
 12  strength_xp         808 non-null    int64 
 13  hitpoints_rank      808 non-null    int64 
 14  hitpoints_level     808 non-null    int64 
 15  hitpoints_xp        808 non-null    int64 
 16  ranged_rank         808 no

## 4. Select skill level attributes

The Hiscores dataset contains rank, level and experience values for each skill.

In this notebook, we select only the columns ending with `_level`.

Reason:

- levels are easier to interpret than raw XP,
- all skill levels are on a similar scale,
- the result is suitable for a first clustering experiment.

In [6]:
level_columns = [column for column in df.columns if column.endswith("_level")]

level_columns

['overall_level',
 'attack_level',
 'defence_level',
 'strength_level',
 'hitpoints_level',
 'ranged_level',
 'prayer_level',
 'magic_level',
 'cooking_level',
 'woodcutting_level',
 'fletching_level',
 'fishing_level',
 'firemaking_level',
 'crafting_level',
 'smithing_level',
 'mining_level',
 'herblore_level',
 'agility_level',
 'thieving_level',
 'slayer_level',
 'farming_level',
 'runecraft_level',
 'hunter_level',
 'construction_level',
 'sailing_level']

In [7]:
df_levels = df[["player"] + level_columns].copy()

df_levels.head()

,player,overall_level,attack_level,defence_level,strength_level,hitpoints_level,ranged_level,prayer_level,magic_level,cooking_level,...,mining_level,herblore_level,agility_level,thieving_level,slayer_level,farming_level,runecraft_level,hunter_level,construction_level,sailing_level
0,Obbyy,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,99
1,Obby Cape,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,99
2,Obby Apples,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,1
3,Obby Kenobi,1752,1,1,99,90,65,31,60,99,...,75,75,75,75,55,85,70,85,75,99
4,obbE x,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,99


## 5. Rename columns

The original column names contain the `_level` suffix.

For easier analysis, we remove this suffix and keep only the skill names.

In [8]:
df_levels.columns = [
    column.replace("_level", "") if column != "player" else column
    for column in df_levels.columns
]

df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,mining,herblore,agility,thieving,slayer,farming,runecraft,hunter,construction,sailing
0,Obbyy,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,99
1,Obby Cape,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,99
2,Obby Apples,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,1
3,Obby Kenobi,1752,1,1,99,90,65,31,60,99,...,75,75,75,75,55,85,70,85,75,99
4,obbE x,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99,99,99,99


## 6. Basic data quality checks

Before feature engineering, we inspect the dataset for:

- missing values,
- invalid values,
- duplicate player names.

This is important because clustering algorithms are sensitive to noisy or inconsistent input data.

In [9]:
df_levels.isna().sum()

player          0
overall         0
attack          0
defence         0
strength        0
hitpoints       0
ranged          0
prayer          0
magic           0
cooking         0
woodcutting     0
fletching       0
fishing         0
firemaking      0
crafting        0
smithing        0
mining          0
herblore        0
agility         0
thieving        0
slayer          0
farming         0
runecraft       0
hunter          0
construction    0
sailing         0
dtype: int64

In [10]:
(df_levels == -1).sum()

player          0
overall         0
attack          0
defence         0
strength        0
hitpoints       0
ranged          0
prayer          0
magic           0
cooking         0
woodcutting     0
fletching       0
fishing         0
firemaking      0
crafting        0
smithing        0
mining          0
herblore        0
agility         0
thieving        0
slayer          0
farming         0
runecraft       0
hunter          0
construction    0
sailing         0
dtype: int64

In Hiscores data, `-1` can indicate that a value is not ranked or not available.

For skill level values this should be rare, but it is still safer to check and handle it.

In [11]:
df_levels = df_levels.replace(-1, np.nan)

df_levels.isna().sum()

player          0
overall         0
attack          0
defence         0
strength        0
hitpoints       0
ranged          0
prayer          0
magic           0
cooking         0
woodcutting     0
fletching       0
fishing         0
firemaking      0
crafting        0
smithing        0
mining          0
herblore        0
agility         0
thieving        0
slayer          0
farming         0
runecraft       0
hunter          0
construction    0
sailing         0
dtype: int64

## 7. Handle missing values

For this first version of the project, rows with missing values are removed.

This keeps the dataset simple and avoids introducing artificial player profiles through imputation.

In [12]:
df_levels = df_levels.dropna()

df_levels.shape

(808, 26)

## 8. Remove duplicate players

Each row should represent exactly one unique OSRS player.

Duplicate rows could distort similarity calculations and clustering results.

In [13]:
df_levels["player"].duplicated().sum()

0

In [14]:
df_levels = df_levels.drop_duplicates(subset="player")

df_levels.shape

(808, 26)

## 9. Create skill group features

The original skill columns describe individual OSRS skills.

To make the player profiles easier to interpret, we create aggregated skill group features.

These engineered features are not required for clustering, but they help with later interpretation of the clusters.

In [15]:
combat_skills = [
    "attack",
    "strength",
    "defence",
    "hitpoints",
    "ranged",
    "prayer",
    "magic"
]

gathering_skills = [
    "mining",
    "fishing",
    "woodcutting",
    "hunter",
    "sailing"
]

production_skills = [
    "cooking",
    "fletching",
    "firemaking",
    "crafting",
    "smithing",
    "herblore",
    "runecraft",
    "construction"
]

support_skills = [
    "agility",
    "thieving",
    "slayer",
    "farming"
]

In [16]:
df_levels["combat_score"] = df_levels[combat_skills].mean(axis=1)
df_levels["gathering_score"] = df_levels[gathering_skills].mean(axis=1)
df_levels["production_score"] = df_levels[production_skills].mean(axis=1)
df_levels["support_score"] = df_levels[support_skills].mean(axis=1)

df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,slayer,farming,runecraft,hunter,construction,sailing,combat_score,gathering_score,production_score,support_score
0,Obbyy,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99.000000,99.0,99.00,99.0
1,Obby Cape,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99.000000,99.0,99.00,99.0
2,Obby Apples,2278,99,99,99,99,99,99,99,99,...,99,99,99,99,99,1,99.000000,79.4,99.00,99.0
3,Obby Kenobi,1752,1,1,99,90,65,31,60,99,...,55,85,70,85,75,99,49.571429,91.4,82.25,72.5
4,obbE x,2376,99,99,99,99,99,99,99,99,...,99,99,99,99,99,99,99.000000,99.0,99.00,99.0


## 10. Create general progression features

In addition to skill group scores, we create general progression indicators:

- average skill level,
- highest skill level,
- lowest skill level,
- skill level standard deviation.

The standard deviation can indicate whether a player has a balanced or specialized account.

In [17]:
base_skill_columns = [
    column for column in df_levels.columns
    if column not in [
        "player",
        "overall",
        "combat_score",
        "gathering_score",
        "production_score",
        "support_score"
    ]
]

In [18]:
df_levels["average_skill_level"] = df_levels[base_skill_columns].mean(axis=1)
df_levels["max_skill_level"] = df_levels[base_skill_columns].max(axis=1)
df_levels["min_skill_level"] = df_levels[base_skill_columns].min(axis=1)
df_levels["skill_level_std"] = df_levels[base_skill_columns].std(axis=1)

df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
0,Obbyy,2376,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000
1,Obby Cape,2376,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000
2,Obby Apples,2278,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.00,99.0,94.916667,99,1,20.004166
3,Obby Kenobi,1752,1,1,99,90,65,31,60,99,...,75,99,49.571429,91.4,82.25,72.5,73.000000,99,1,27.700259
4,obbE x,2376,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000


## 11. Inspect the cleaned dataset

Now we inspect the cleaned and enriched dataset with descriptive statistics.

This helps us understand the scale and distribution of the selected attributes before applying similarity and clustering methods.

In [19]:
df_levels.describe()

,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,woodcutting,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
count,808.000000,808.000000,808.000000,808.000000,808.000000,808.00000,808.000000,808.000000,808.000000,808.000000,...,808.000000,808.000000,808.000000,808.000000,808.000000,808.000000,808.000000,808.000000,808.000000,808.000000
mean,2012.986386,87.164604,85.835396,90.073020,89.943069,89.32302,83.118812,89.051980,88.163366,86.230198,...,83.039604,49.772277,87.787129,77.949010,84.544090,84.632735,84.130776,96.189356,46.944307,11.920510
std,591.108224,26.902345,28.425844,25.002817,25.083291,25.76908,26.515773,25.876214,24.097140,25.220200,...,25.891604,44.016868,24.477534,24.702534,24.168268,25.303188,23.211706,11.784834,42.739035,10.676578
min,0.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000
25%,1980.000000,91.000000,90.000000,99.000000,99.000000,98.00000,80.000000,96.000000,90.000000,85.000000,...,83.000000,1.000000,91.857143,69.400000,83.187500,84.187500,82.500000,99.000000,1.000000,0.612372
50%,2254.000000,99.000000,99.000000,99.000000,99.000000,99.00000,94.000000,99.000000,99.000000,99.000000,...,93.000000,55.000000,98.142857,83.000000,94.500000,95.500000,93.916667,99.000000,51.000000,10.356518
75%,2369.000000,99.000000,99.000000,99.000000,99.000000,99.00000,99.000000,99.000000,99.000000,99.000000,...,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,98.708333,99.000000,94.250000,19.899806
max,2376.000000,99.000000,99.000000,99.000000,99.000000,99.00000,99.000000,99.000000,99.000000,99.000000,...,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,99.000000,49.095884


In [20]:
df_levels.head()

,player,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
0,Obbyy,2376,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000
1,Obby Cape,2376,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000
2,Obby Apples,2278,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.00,99.0,94.916667,99,1,20.004166
3,Obby Kenobi,1752,1,1,99,90,65,31,60,99,...,75,99,49.571429,91.4,82.25,72.5,73.000000,99,1,27.700259
4,obbE x,2376,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000


## 12. Prepare feature matrix preview

For later similarity calculation and clustering, we need a numeric feature matrix.

The `player` column is an identifier, therefore it will not be used as a model input.

In [21]:
feature_columns = [
    column for column in df_levels.columns
    if column != "player"
]

X = df_levels[feature_columns]

X.head()

,overall,attack,defence,strength,hitpoints,ranged,prayer,magic,cooking,woodcutting,...,construction,sailing,combat_score,gathering_score,production_score,support_score,average_skill_level,max_skill_level,min_skill_level,skill_level_std
0,2376,99,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000
1,2376,99,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000
2,2278,99,99,99,99,99,99,99,99,99,...,99,1,99.000000,79.4,99.00,99.0,94.916667,99,1,20.004166
3,1752,1,1,99,90,65,31,60,99,99,...,75,99,49.571429,91.4,82.25,72.5,73.000000,99,1,27.700259
4,2376,99,99,99,99,99,99,99,99,99,...,99,99,99.000000,99.0,99.00,99.0,99.000000,99,99,0.000000


In [22]:
X.shape

(808, 33)

## 13. Save cleaned dataset

The cleaned dataset is saved into the `data/processed` folder.

This file will be used by the next notebooks:

- similarity and scaling analysis,
- clustering analysis.

In [23]:
df_levels.to_csv(output_path, index=False, encoding="utf-8")

output_path

WindowsPath('C:/Projects/osrs-player-segmentation/data/processed/osrs_hiscores_cleaned.csv')

## Summary

In this notebook, the collected OSRS Hiscores dataset was cleaned and prepared for further analysis.

The main steps were:

- loading the collected Hiscores dataset,
- selecting skill level attributes,
- checking invalid and missing values,
- removing duplicate players,
- creating aggregated skill group features,
- creating general progression indicators,
- saving the cleaned dataset.

The next notebook will focus on similarity, dissimilarity, scaling and distance-based analysis.